# Introduction to **TabPFN** and **TabICL**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/drive/1F0hKCoyl21XL3RL-cR6FXhvxasd4ihei?usp=sharing)

Authors: [Pedro L. C. Rodrigues](https://plcrodrigues.github.io) and [Thomas Moreau](https://tommoral.github.io)

- **TabPFN** : Hollman et al. "Accurate predictions on small data with a tabular foundation model" (2025) [[link](https://www.nature.com/articles/s41586-024-08328-6)]

- **TabICL** : Qu et al. "TabICLv2: A better, faster, scalable, and open tabular foundation model" (2026) [[link](https://arxiv.org/abs/2602.11139)]

For a very nice overview on the general topic of tabular foundation models, take a look at this online book :

- Christoph Molnar "Tabular Foundation Models: A short and opinionated guide" (2026) [link](https://tabularfoundationmodels.com/)

For a lightweight and educational reimplementation of TabPFN, check [nanoTabPFN](https://github.com/automl/nanoTabPFN).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

%matplotlib inline

The Python implementation of TabPFN is developed by people from [Prior Labs](https://docs.priorlabs.ai/overview) and follows the same API from `scikit-learn`. 

Note, however, that to use the last version of the TabPFN's foundational model, you will need to authenticate at HuggingFace, which can be a bit messy. Because of this, we will be focusing on TabPFN-V2, which should be more than enough. 

⚡ GPU Recommended: For optimal performance, use a GPU (even older ones with ~8GB VRAM work well; 16GB needed for some large datasets). On CPU, only small datasets (≲1000 samples) are feasible. Note that **this notebook can be run on Codalab with the top button**.

First of all, you will need to install the package by as below

In [ ]:
!pip install -U tabpfn

# 1 Regression with TabPFN

We investigate how TabPFN can be used for regression and compare his performance versus other classic regressors.

In [ ]:
from sklearn.utils import shuffle
from sklearn.model_selection import KFold, cross_val_score
from sklearn.datasets import fetch_california_housing, load_diabetes

from tabpfn import TabPFNRegressor
from tabpfn.constants import ModelVersion

import pandas as pd, requests

# Loading the Boston dataset
cols = ["CRIM","ZN","INDUS","CHAS","NOX","RM","AGE","DIS","RAD","TAX","PTRATIO","B","LSTAT","MEDV"]
df_boston = pd.read_csv("https://archive.ics.uci.edu/ml/machine-learning-databases/housing/housing.data",
                        sep='\\s+', header=None, names=cols)

print(df_boston.shape)
df_boston.head()

In [ ]:
# Convert to pure numpy arrays
X, y = df_boston.drop(columns=["MEDV"]).values, df_boston["MEDV"].values

# Choose cross-validation strategy
cv = KFold(shuffle=True, n_splits=5)

# Instantiate TabPFN for regression
# regressor_tabpfn = TabPFNRegressor()
regressor_tabpfn = TabPFNRegressor.create_default_for_version(ModelVersion.V2)
regressor_tabpfn.n_estimators = 1

scores = []
for idx_train, idx_test in tqdm(cv.split(X, y)):
    X_train, y_train = X[idx_train], y[idx_train]
    X_test, y_test = X[idx_test], y[idx_test]
    regressor_tabpfn.fit(X_train, y_train)
    scores.append(regressor_tabpfn.score(X_test, y_test))
print(np.mean(scores))

<div class="alert alert-success">
    <b>EXERCISE</b>:
     <ul>
         <li>What is happening in the <span><code>fit</code></span> method?</li>
     </ul>
</div>

Let's see now how a `RandomForestRegressor` and the `HistGradientRegressor` behave

In [ ]:
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
regressor_rf = RandomForestRegressor()
regressor_hgbr = HistGradientBoostingRegressor()
est_dict = {'rf':regressor_rf, 'hgbr':regressor_hgbr}
for key, value in est_dict.items():
    scores = cross_val_score(value, X, y, cv=cv)
    print(key, np.mean(scores))

We see that TabPFN beats both baseslines by quite a margin. However, it took much more time...

Let's consider now a different dataset.

In [ ]:
from sklearn.datasets import fetch_california_housing
df_california, targets = fetch_california_housing(return_X_y=True, as_frame=True)
print(df_california.shape)
df_california.head()

The dataset is bigger than the previous one, so let's see how TabPFN behaves.

In [ ]:
from sklearn.model_selection import train_test_split # let's avoid cross-val for time sake
X, y = df_california.values, targets.values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)
regressor_tabpfn.fit(X_train, y_train)
print(regressor_tabpfn.score(X_test, y_test))

<div class="alert alert-success">
    <b>EXERCISE</b>:
     <ul>
         <li>Why do you think we exploded in memory?</li>
     </ul>
</div>

One possible trick is to subsample the dataset and use an ensemble of TabPFN regressors as below.

In [ ]:
regressor_tabpfn.ignore_pretraining_limits = True
regressor_tabpfn.n_estimators = 1
regressor_tabpfn.inference_config = {"SUBSAMPLE_SAMPLES": 500}
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)
regressor_tabpfn.fit(X_train, y_train)
print(regressor_tabpfn.score(X_test, y_test))

Or even better, use a GPU :-) 

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/drive/1F0hKCoyl21XL3RL-cR6FXhvxasd4ihei?usp=sharing)


In [ ]:
regressor_rf = RandomForestRegressor()
regressor_hgbr = HistGradientBoostingRegressor()
est_dict = {'rf':regressor_rf, 'hgbr':regressor_hgbr}
for key, est in est_dict.items():
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)
    est.fit(X_train, y_train)
    score = est.score(X_test, y_test)
    print(key, score)

In the slides, we saw that **TabICL** can, in principle, scale to any number of samples, due to the way that rows and columns are embedded in its architecture. 

So should we try to use it?

In [ ]:
!pip install -U tabicl # watch out for the python version!

We will consider the same dataset as before, but with a **varying number of training points given as context**: 50, 100, 200, ... up to 5,000. 

The test set is fixed (2,000 points), so the only thing that changes between runs is how much data each model sees.

We compare the two foundation models with the two tree ensembles. 

Remember that for TabPFN and TabICL `fit` does nothing but store the data: all the work happens at `predict` time, inside the transformer.

> ⏱️ On a CPU, the 5,000-point run takes a few minutes for TabPFN. Shorten `n_grid` if you are in a hurry, or run this cell on Colab with a GPU.


In [ ]:
import time
from tabicl import TabICLRegressor

rng = np.random.default_rng(0)
perm = rng.permutation(len(X))
X_pool, y_pool = X[perm[:-2000]], y[perm[:-2000]]   # we draw the context from here
X_test, y_test = X[perm[-2000:]], y[perm[-2000:]]   # fixed test set

n_grid = [50, 100, 200, 500, 1000, 2000, 5000]

def make_models():
    tabpfn = TabPFNRegressor.create_default_for_version(ModelVersion.V2)
    tabpfn.n_estimators = 1
    return {
        "TabPFN": tabpfn,
        "TabICL": TabICLRegressor(n_estimators=1),
        "HistGradientBoosting": HistGradientBoostingRegressor(random_state=0),
        "Random Forest": RandomForestRegressor(random_state=0, n_jobs=-1),
    }

records = []
for n_train in tqdm(n_grid):
    for name, model in make_models().items():
        t0 = time.time()
        model.fit(X_pool[:n_train], y_pool[:n_train])
        r2 = model.score(X_test, y_test)
        records.append({"model": name, "n_train": n_train, "r2": r2, "time": time.time() - t0})

results = pd.DataFrame(records)
results.pivot(index="n_train", columns="model", values="r2").round(3)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, df_m in results.groupby("model"):
    axes[0].plot(df_m["n_train"], df_m["r2"], marker="o", label=name)
    axes[1].plot(df_m["n_train"], df_m["time"], marker="o", label=name)
axes[0].set(xscale="log", xlabel="number of training points (context)", ylabel="test $R^2$",
            title="California housing, 2,000 test points")
axes[1].set(xscale="log", yscale="log", xlabel="number of training points (context)",
            ylabel="fit + predict time (s)")
axes[0].legend()
fig.tight_layout()


<div class="alert alert-success">
    <b>EXERCISE</b>:
     <ul>
         <li>Below which training-set size do the foundation models beat the tree ensembles, and by how much?</li>
         <li>What does this say about their <em>few-shot</em> behaviour?</li>
         <li>Look at the right-hand plot. Where does TabPFN's cost go when the context grows, and why is it different from the tree models?</li>
         <li>The tree ensembles were run with default hyperparameters. Would tuning them (e.g. with the <code>optuna</code> search from Day 2) change the conclusion in the small-<code>n</code> regime? Try it for <code>n_train = 200</code>.</li>
     </ul>
</div>


# 2 Classification with TabPFN and TabICL

Let's switch to classifcation problems. First a toy example, then a small benchmark, and finally a larger dataset.

## 2.1 A toy example: what do the decision boundaries look like?

Before turning to real datasets, let's build some intuition on a 2D toy problem, the classic "two moons" with some noise. In two dimensions, we can plot the predicted probability over the whole plane and *see* the function that each model has learned.

We compare the two foundation models with the tree ensembles that we studied in a previous session.

In [ ]:
from sklearn.datasets import make_moons
from sklearn.inspection import DecisionBoundaryDisplay
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from tabpfn import TabPFNClassifier
from tabicl import TabICLClassifier

X, y = make_moons(n_samples=300, noise=0.3, random_state=0)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=42)

clf_tabpfn = TabPFNClassifier.create_default_for_version(ModelVersion.V2)
clf_tabpfn.n_estimators = 1
models = {
    "TabPFN": clf_tabpfn,
    "TabICL": TabICLClassifier(n_estimators=1),
    "Random Forest": RandomForestClassifier(random_state=0),
    "HistGradientBoosting": HistGradientBoostingClassifier(),
}

fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharex=True, sharey=True)
for ax, (name, clf) in zip(axes, models.items()):
    clf.fit(X_train, y_train)
    # color = predicted probability of the blue class on a grid covering the plane
    DecisionBoundaryDisplay.from_estimator(
        clf, X, response_method="predict_proba", grid_resolution=80,
        cmap="RdBu", alpha=0.8, eps=0.5, ax=ax,
    )
    ax.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap="RdBu", edgecolors="k", s=20)
    ax.set_title(f"{name}\ntest accuracy = {clf.score(X_test, y_test):.2f}")
fig.tight_layout()

All four models reach a similar test accuracy, but the functions they learned are very different:

- **Random Forest** and **HistGradientBoosting** are built from axis-aligned splits, so their decision boundaries are made of horizontal and vertical steps, and far from the data they simply extend the outermost splits.
- **TabPFN** and **TabICL** produce smooth, curved boundaries that follow the shape of the moons. Nobody told them that the classes are separated by smooth curves: this comes from the prior over synthetic datasets that they were pre-trained on, which acts as an inductive bias when they predict in context.

## 2.2 Classification on the Titanic dataset

Let's consider the classic Titanic dataset.

In [ ]:
from sklearn.datasets import fetch_openml
df, target = fetch_openml("titanic", version=1, as_frame=True, return_X_y=True, parser="liac-arff")
df = df.drop(columns='boat')
X, y = df.values, target.values
df.head()

We saw in previous classes that we can not simply plug-in the Titanic dataset into standard scikit-learn estimators. 

First, it is necessary to pre-process the data, encode categorical features, etc. 

Is it same for TabPFN ?

In [ ]:
from sklearn.model_selection import StratifiedKFold
cv = StratifiedKFold(shuffle=True, n_splits=5)

# Instantiate TabPFN for classification
from tabpfn import TabPFNClassifier
clf_tabpfn = TabPFNClassifier.create_default_for_version(ModelVersion.V2)
clf_tabpfn.n_estimators = 1

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)
clf_tabpfn.fit(X_train, y_train)
print(clf_tabpfn.score(X_test, y_test))

What about TabICL ?

In [ ]:
from tabicl import TabICLClassifier
clf_icl = TabICLClassifier(n_estimators=1)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)
clf_icl.fit(X_train, y_train)
print(clf_icl.score(X_test, y_test))

The documention can help us : https://github.com/soda-inria/tabicl?tab=readme-ov-file#basic-integration

In [ ]:
from skrub import TableVectorizer
from tabicl import TabICLClassifier
from sklearn.pipeline import make_pipeline

pipeline = make_pipeline(
    TableVectorizer(),  # automatically handles various data types
    TabICLClassifier(n_estimators=1)  # beware of the default parameters!
)

X_train, X_test, y_train, y_test = train_test_split(df, target, test_size=0.33, random_state=42) # note that we pass the dataframe!
pipeline.fit(X_train, y_train)
print(pipeline.score(X_test, y_test))

<div class="alert alert-success">
    <b>EXERCISE</b>:
     <ul>
         <li>Why can TabPFN preprocess categorical features directly and TabICL needs a pipeline?</li>
     </ul>
</div>

In [ ]:
from skrub import tabular_pipeline
est = tabular_pipeline('classifier')
X_train, X_test, y_train, y_test = train_test_split(df, target, test_size=0.33, random_state=42) # note that we pass the dataframe!
est.fit(X_train, y_train)
print(est.score(X_test, y_test))

Let's consider now a larger dataset and see how **TabICL** behaves.

## 2.3 Classification on the adult census dataset

In [ ]:
import pandas as pd
from pathlib import Path
from urllib.request import urlretrieve

DATA_DIR = Path().parent / "data"

url = ("https://archive.ics.uci.edu/ml/machine-learning-databases"
       "/adult/adult.data")

local_filename =  DATA_DIR/ Path(url).name
if not local_filename.exists():
    print("Downloading Adult Census datasets from UCI")
    DATA_DIR.mkdir(exist_ok=True)
    urlretrieve(url, local_filename)

names = ("age, workclass, fnlwgt, education, education-num, "
         "marital-status, occupation, relationship, race, sex, "
         "capital-gain, capital-loss, hours-per-week, "
         "native-country, income").split(', ') 
df = pd.read_csv(local_filename, names=names)
df = df.rename(columns={'income': 'class'})

columns_to_plot = [
    "age",
    "education-num",
    "capital-loss",
    "capital-gain",
    "hours-per-week",
    "class",
]
df = df[columns_to_plot]
print(df.shape)
df.head()

In [ ]:
target_name = "class"
target = df[target_name]
X, y = df.iloc[:,:-1].values, target.values
y = (y == ' <=50K').astype(np.int8)

In [ ]:
clf_icl = TabICLClassifier(n_estimators=4)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)
clf_icl.fit(X_train, y_train)
print(clf_icl.score(X_test, y_test))

Even though TabICL "scales to any number of samples", its attention memory still grows with the square of the number of rows. Indeed, when going through the transformer, the 32,561 rows (21,815 train + 10,746 test) are compared with each of the 21,815 training rows, in each of 8 attention heads. Stored as float32 (4 bytes per number), that is 8 × 32,561 × 21,815 × 4 bytes = 21.17 GiB, exactly the size reported in the error.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

clf_hgbr = HistGradientBoostingClassifier()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)
clf_hgbr.fit(X_train, y_train)
print(clf_hgbr.score(X_test, y_test))

# 3 Univariate regression with uncertainty using TFMs

So far we only looked at point predictions. But tabular foundation models do not output a single number: for each test point, they predict the whole **distribution** over the target, $p(y \mid x, \mathcal{D}_{\text{train}})$, aka the "posterior predictive distribution" (PPD). Uncertainty quantification therefore comes almost for free.

We use the [Bike Sharing dataset](https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset), which contains the number of bikes rented every hour in Washington D.C. in 2011 and 2012. 

So that we can plot everything, we use a single feature, the **hour of the day**, to predict the **number of rentals**.

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
from zipfile import ZipFile

DATA_DIR = Path().parent / "data"

url = "https://archive.ics.uci.edu/static/public/275/bike+sharing+dataset.zip"

local_filename = DATA_DIR / "bike_sharing.zip"
if not local_filename.exists():
    print("Downloading Bike Sharing dataset from UCI")
    DATA_DIR.mkdir(exist_ok=True)
    urlretrieve(url, local_filename)

with ZipFile(local_filename) as z:
    df_bike = pd.read_csv(z.open("hour.csv"))
print(df_bike.shape)
df_bike.head()

We give the models 1,000 hours as training set (i.e. as context), and keep 2,000 other hours as test set to check the quality of the intervals.

In [ ]:
from sklearn.model_selection import train_test_split

X, y = df_bike[["hr"]].values, df_bike["cnt"].values
X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=1000, test_size=2000, random_state=0
)

hours = np.arange(24).reshape(-1, 1)  # where we evaluate the predictive distribution
alphas = [0.05, 0.5, 0.95]  # 5% quantile, median, 95% quantile

-- **With TabICL**

The method `TabICLRegressor.predict` accepts `output_type="quantiles"` together with the probability levels `alphas`. 

It returns an array of shape `(n_samples, n_quantiles)`.

In [ ]:
from tabicl import TabICLRegressor

reg_icl = TabICLRegressor(n_estimators=4)
reg_icl.fit(X_train, y_train)
q_icl = reg_icl.predict(hours, output_type="quantiles", alphas=alphas)
q_icl.shape

**-- With TabPFN**

It is just as easy with TabPFN: the argument is named `quantiles` instead of `alphas`, and the output is a list with one array per quantile, which we stack to get the same shape as for TabICL.

In [ ]:
from tabpfn import TabPFNRegressor
from tabpfn.constants import ModelVersion

reg_pfn = TabPFNRegressor.create_default_for_version(ModelVersion.V2)
reg_pfn.n_estimators = 4
reg_pfn.fit(X_train, y_train)
q_pfn = np.column_stack(reg_pfn.predict(hours, output_type="quantiles", quantiles=alphas))
q_pfn.shape

Let's plot the median and the 90% prediction interval (between the 5% and 95% quantiles) for each hour of the day.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
jitter = np.random.default_rng(0).uniform(-0.3, 0.3, len(X_train))  # to see overlapping points
for ax, (name, q) in zip(axes, {"TabICL": q_icl, "TabPFN": q_pfn}.items()):
    ax.scatter(X_train[:, 0] + jitter, y_train, s=5, alpha=0.3, color="k", label="training data")
    ax.fill_between(hours[:, 0], q[:, 0], q[:, 2], alpha=0.3, label="90% interval")
    ax.plot(hours[:, 0], q[:, 1], lw=2, label="median")
    ax.set(title=name, xlabel="hour of the day")
axes[0].set_ylabel("number of rentals")
axes[0].legend();

Both models capture the two commuting peaks (8am and 5-6pm). More importantly, **the intervals adapt to the data**: they are very narrow at night, when almost nobody rents a bike, and very wide at rush hours. Note that a constant-width interval, such as *prediction ± 1.645 × standard deviation of the residuals*, would be far too wide at night and far too narrow at 8am.

Are these intervals reliable? If they are well calibrated, about 90% of the **test** points should fall inside them.

In [ ]:
q_icl_test = reg_icl.predict(X_test, output_type="quantiles", alphas=alphas)
q_pfn_test = np.column_stack(reg_pfn.predict(X_test, output_type="quantiles", quantiles=alphas))

for name, q in {"TabICL": q_icl_test, "TabPFN": q_pfn_test}.items():
    coverage = np.mean((y_test >= q[:, 0]) & (y_test <= q[:, 2]))
    print(f"{name}: {coverage:.1%} of the test points are inside the 90% interval")

We are close to the nominal 90%, without any tuning, and without ever training a model on this dataset.

### Looking at the full predictive distribution

Look again at 8am in the plot above: the median is around 300 rentals, yet very few training points are close to 300. The quantiles hide part of the story, so let's look at the whole predictive distribution. The two models represent it differently:

- **TabICL** directly predicts a large set of quantiles (999 of them, accessible with `output_type="raw_quantiles"`). From many quantiles we can approximate the density: between two consecutive quantiles, there is a known probability mass spread over a known width.
- **TabPFN** predicts a histogram with 5,000 bins, called a *bar distribution*. With `output_type="full"`, we get the `logits` of each bin (a softmax turns them into probabilities) and the bin edges in `criterion.borders`.

In [ ]:
x_8am = np.array([[8]])

# TabICL: probability mass between consecutive quantiles / distance between them
levels = np.linspace(0.01, 0.99, 99)
q = reg_icl.predict(x_8am, output_type="quantiles", alphas=levels)[0]
density_icl = np.diff(levels) / np.diff(q)
y_icl = (q[1:] + q[:-1]) / 2

# TabPFN: probability of each bin / width of the bin
full = reg_pfn.predict(x_8am, output_type="full")
probs = full["logits"].softmax(dim=-1)[0].cpu().numpy()
borders = full["criterion"].borders.cpu().numpy()
density_pfn = probs / np.diff(borders)
y_pfn = (borders[1:] + borders[:-1]) / 2

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(y_train[X_train[:, 0] == 8], bins=30, density=True, color="lightgray",
        label="training data at 8am")
ax.plot(y_icl, density_icl, lw=2, label="TabICL")
ax.plot(y_pfn, density_pfn, lw=2, label="TabPFN")
ax.set(xlim=(-50, 1000), xlabel="number of rentals at 8am", ylabel="density")
ax.legend();

Both models predict a **bimodal** distribution at 8am: one mode around 100 rentals and a broad bump between 400 and 750. The median falls in the valley between the two, where very few observations actually are. A model that only outputs a point prediction, or even a mean and a standard deviation, could not reveal this.

<div class="alert alert-success">
    <b>EXERCISE</b>:
     <ul>
         <li>Where does this bimodality at 8am come from? Add to <code>X</code> a second column of <code>df_bike</code> that you think explains it, refit TabICL, and look at the 90% interval at 8am for the different values of this new feature.</li>
     </ul>
</div>